In [1]:
%load_ext lab_black
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import polars as pl
import time
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.preprocessing import OrdinalEncoder
import os
import gc
import matplotlib.pyplot as plt
import warnings
import sys
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset
import random

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
from merlin.loader.torch import Loader
from merlin.io import Dataset as Dataset_merlin

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")
print(device)

sys.path.append("../src/otto_recommender")
import utils
import similar

warnings.filterwarnings("ignore")
tqdm.pandas()

pd.set_option("max_rows", 500)

%matplotlib inline

cuda
INFO: Pandarallel will run on 16 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.


In [2]:
def seed_everything(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False  # set True to be faster
    # torch.use_deterministic_algorithms(True)
    print(f"Setting all seeds to be {seed} to reproduce...")


seed_everything(42)

Setting all seeds to be 42 to reproduce...


In [3]:
class Config:
    col_user = "user"
    col_aid = "aid"
    col_type = "type"
    col_n_session = "n_session"
    col_n_day = "n_day"
    col_n_week = "n_week"
    col_dayofweek = "dayofweek"
    col_hour = "hour"
    col_ts = "ts"
    col_num_action = "num_action"
    col_train_seq = "train_seq"
    col_target = "target"
    col_type_group = "type_group"
    path_data = "../data/02_intermediate/"
    batch_size_train = 512
    batch_size_infer = 512
    n_epochs = 10
    max_len = 10
    model_name = "def_tune_"
    hidden_dim = 128
    embedding_dim = 100
    init_lr = 0.0005
    cur_model_path = "def_tune__epoch11_larger_lr_and_bs.pth"
    cur_model_output_path = "../data/07_model_output/"
    num_recs = 100
    new_split = "_new_split"


config = Config()

In [4]:
%%time

X_train, X_val, X_val_seq, ordinal_encoder = utils.get_train_val_dfs(config, do_ordinal_encoder_transform=False, drop_pad=True, new_split=True)
# X_train, X_val, X_val_seq, ordinal_encoder = utils.dfs_interactions_to_list(
#     config,
#     X_train,
#     X_val,
#     X_val_seq,
#     ordinal_encoder,
#     return_only_val=False,
#     truncate_val=False,
#     return_just_splitted=True,
# )
pad = ordinal_encoder.unknown_value

CPU times: user 8min 47s, sys: 1min 53s, total: 10min 41s
Wall time: 2min 16s


In [5]:
del X_train, X_val, X_val_seq
gc.collect()

190

In [6]:
%%time

PATH_DATA = config.path_data
df_train = pl.read_parquet(PATH_DATA + 'df_train_n_session.parquet')
df_test = pl.read_parquet(PATH_DATA + 'df_test_n_session.parquet')

test_users = df_test[config.col_user].unique()
df = pl.concat([df_train, df_test])
df = df.sort(config.col_ts, reverse=True)

df = df.to_pandas()
df[config.col_num_action] = df.groupby(config.col_user)[config.col_aid].cumcount()
df[config.col_aid] = ordinal_encoder.transform(
            df[config.col_aid].values.reshape(-1, 1)
        ).astype(int)
df = df[df[config.col_aid] != pad]
df = pl.from_pandas(df)

df = df.with_columns(
    [
        pl.col(config.col_num_action)
        .cast(pl.Int32, strict=False)
        .alias(config.col_num_action),
        pl.col(config.col_n_session)
        .cast(pl.Int32, strict=False)
        .alias(config.col_n_session),
    ]
)

del df_train, df_test
gc.collect()

df_inter2inter = (
    df.groupby([config.col_user])
    .agg(
        [
            pl.col(config.col_aid).list(),
        ]
    )
)

data = list(df_inter2inter.to_pandas()[config.col_aid])

del df_inter2inter, df
gc.collect()

CPU times: user 10min 52s, sys: 1min 24s, total: 12min 17s
Wall time: 3min 14s


0

In [7]:
%%time

PATH_DATA = config.path_data
df_test = pl.read_parquet(PATH_DATA + "df_test_n_session.parquet")
df_test = df_test.sort(config.col_ts)
df_test = df_test.to_pandas()
df_test[config.col_aid] = ordinal_encoder.transform(
    df_test[config.col_aid].values.reshape(-1, 1)
).astype(int)
df_test = df_test[df_test[config.col_aid] != pad]
df_test = pl.from_pandas(df_test)
df_test = df_test.groupby([config.col_user]).agg(
    [
        pl.col(config.col_aid).list().alias(config.col_train_seq),
    ]
)

data_to_predict = df_test.to_pandas()[config.col_train_seq].tolist()

CPU times: user 12.4 s, sys: 1.74 s, total: 14.2 s
Wall time: 4.31 s


In [8]:
%%time

for mod in similar.recommenders:
    print(mod)
    model_similar = similar.train_similar_model(data=data, n_items=pad, recommeder_cls=mod, max_neighbours=100)
    recommender = similar.SimilarRecommender(implicit_model=model_similar)

    recs = recommender.recommend(data_to_predict, n_recs=config.num_recs, last_k_items=10)

    df_idx = recs.to_pandas()[["index"]].drop_duplicates().reset_index(drop=True)
    df_idx["user"] = df_test.to_pandas()["user"].astype(int)
    df_idx = pl.from_pandas(df_idx)
    recs = recs.join(df_idx, on="index")

    recs = recs.to_pandas()
    recs['item_id'] = ordinal_encoder.inverse_transform(
            recs["item_id"].values.reshape(-1, 1)
        ).reshape(1, -1)[0]
    recs = recs.dropna()
    recs = pl.from_pandas(recs)

    recs = recs.rename({'similar_score': f'{mod}_score', 'similar_rank': f'{mod}_rank'})

    recs.write_parquet(config.cur_model_output_path + f'cur_{mod}_df_test_preds_{config.num_recs}.parquet')

#     recs = (
#         recs.groupby(["index"])
#         .agg(
#             [
#                 pl.col("item_id").list(),
#             ]
#         )
#         .sort("index")
#         .to_pandas()
#     )

#     recs["user"] = X_val_seq.to_pandas()["user"]
#     recs["item_id"] = recs["item_id"].progress_apply(lambda x: utils.truncate_list(list(x), from_start=True, max_len=20))
#     recs = recs.merge(X_val[["user", "target"]].to_pandas())

#     preds = recs["item_id"].tolist()
#     preds = [list(pred) for pred in preds]

#     targets = recs["target"].tolist()
#     targets = [list(target) for target in targets]

#     print(utils.recall_several_targets(preds, targets))
    print()
    print('_' * 10)

bm25


  0%|          | 0/954978 [00:00<?, ?it/s]

predicting:   0%|          | 0/1633244 [00:00<?, ?it/s]


__________
tfidf


  0%|          | 0/954978 [00:00<?, ?it/s]

predicting:   0%|          | 0/1633244 [00:00<?, ?it/s]


__________
cosine


  0%|          | 0/954978 [00:00<?, ?it/s]

predicting:   0%|          | 0/1633244 [00:00<?, ?it/s]


__________
CPU times: user 1h 51min 4s, sys: 4min 38s, total: 1h 55min 43s
Wall time: 1h 7min 32s
